# Imports

In [1]:
import numpy as np
import joblib

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

# Load Feature Extractor (AlexNet-style)

In [2]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

print("_/_/ VGG16 (AlexNet-style) feature extractor ready")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 129s 2us/step
_/_/ VGG16 (AlexNet-style) feature extractor ready


# Data Generators

In [3]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

validation_generator = datagen.flow_from_directory(
    'dataset/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    'dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 1145 images belonging to 6 classes.
Found 267 images belonging to 6 classes.
Found 220 images belonging to 6 classes.


# Feature Extraction Function

In [4]:
def extract_features(generator, model):
    features, labels = [], []

    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        f_batch = model.predict(x_batch, verbose=0)

        features.append(f_batch)
        labels.append(np.argmax(y_batch, axis=1))

    return np.vstack(features), np.hstack(labels)

# Extract Features

In [5]:
print("Extracting TRAIN features...")
X_train, y_train = extract_features(train_generator, feature_extractor)

print("Extracting VALIDATION features...")
X_val, y_val = extract_features(validation_generator, feature_extractor)

print("Extracting TEST features...")
X_test, y_test = extract_features(test_generator, feature_extractor)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Extracting TRAIN features...
Extracting VALIDATION features...
Extracting TEST features...
Shapes: (1145, 512) (267, 512) (220, 512)


# Normalize Features

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("_/_/ Features normalized")

_/_/ Features normalized


# PCA (CRITICAL for KNN)

In [7]:
pca = PCA(n_components=150)

X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)

print("_/ PCA applied:", X_train.shape)

_/ PCA applied: (1145, 150)


# Train KNN

In [49]:
print("Training KNN...")

knn_model = KNeighborsClassifier(
    n_neighbors=93,
    weights='distance',
    metric='euclidean'
)

knn_model.fit(X_train, y_train)

print("_/ KNN trained")

Training KNN...
_/ KNN trained


# Validation Evaluation

In [50]:
y_val_pred = knn_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

Validation Accuracy: 0.8164794007490637


# Test Evaluation

In [2]:
y_test_pred = knn_model.predict(X_test)

print("_/ Final Test Accuracy:", accuracy_score(y_test, y_test_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

NameError: name 'knn_model' is not defined

# Save Models

In [ ]:
joblib.dump(knn_model, "alexnet_knn_model.pkl")
joblib.dump(scaler, "alexnet_knn_scaler.pkl")
joblib.dump(pca, "alexnet_knn_pca.pkl")

print("_/ Model saved")

_/ Model saved


# Single Image Prediction

In [ ]:
from tensorflow.keras.preprocessing import image

knn_model = joblib.load("alexnet_knn_model.pkl")
scaler = joblib.load("alexnet_knn_scaler.pkl")
pca = joblib.load("alexnet_knn_pca.pkl")

class_indices = train_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

img_path = "dataset/test/A2-Sitting-down/235.png"
img = image.load_img(img_path, target_size=(224,224))

img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

features = feature_extractor.predict(img_array)
features = scaler.transform(features)
features = pca.transform(features)

pred = knn_model.predict(features)[0]
confidence = np.max(knn_model.predict_proba(features))

print("Predicted Class:", index_to_class[pred])
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Predicted Class: A2-Sitting-down
Confidence: 0.7002854990642805
